In [9]:
# TASK 1 - Data Processing

"""
MasakhaNEWS — Task 1: Data processing.
Loads eng/xho/sna splits, cleans + tokenises headline+text, and builds
TF-IDF feature matrices for a multinomial logistic regression model.
"""

import os
import re
import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "data"
OUT_DIR = "features"
LANGS = ["eng", "xho", "sna"]
SPLITS = ["train", "dev", "test"]

URL_RE = re.compile(r"https?://\S+|www\.\S+")
DIGIT_RE = re.compile(r"\d+")
TOKEN_RE = re.compile(r"<num>|[a-z]+")  # text is lowercased before this runs


def load_split(lang, split):
    df = pd.read_csv(os.path.join(DATA_DIR, lang, f"{split}.tsv"), sep="\t")
    df["full_text"] = df["headline"].fillna("") + " " + df["text"].fillna("")
    return df[["category", "full_text"]]


def load_language(lang):
    return {split: load_split(lang, split) for split in SPLITS}


def clean_text(text):
    text = text.lower()
    text = URL_RE.sub(" ", text)
    text = DIGIT_RE.sub(" <num> ", text)  # keep "has a number" signal without huge vocab
    return text


def tokenize(text):
    # same simple tokenizer for all 3 languages - no stemmer/stopword list
    # exists for xho/sna so we don't want to give eng special treatment
    return TOKEN_RE.findall(clean_text(text))


def build_vectorizer(method="tfidf"):
    # same cleaning/tokenizing/ngrams for both, so the only thing that
    # differs is count-based vs tfidf-weighted features
    common = dict(
        tokenizer=tokenize,
        lowercase=False,
        token_pattern=None,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
    )
    if method == "tfidf":
        return TfidfVectorizer(sublinear_tf=True, **common)
    elif method == "bow":
        return CountVectorizer(**common)
    else:
        raise ValueError(f"unknown method: {method}")


def process_language(lang, method="tfidf"):
    print(f"=== {lang} ({method}) ===")
    data = load_language(lang)

    vectorizer = build_vectorizer(method)
    X_train = vectorizer.fit_transform(data["train"]["full_text"])  # fit on train only
    X_dev = vectorizer.transform(data["dev"]["full_text"])
    X_test = vectorizer.transform(data["test"]["full_text"])

    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(data["train"]["category"])
    y_dev = label_encoder.transform(data["dev"]["category"])
    y_test = label_encoder.transform(data["test"]["category"])

    out_dir = os.path.join(OUT_DIR, lang, method)
    os.makedirs(out_dir, exist_ok=True)
    sparse.save_npz(os.path.join(out_dir, "X_train.npz"), X_train)
    sparse.save_npz(os.path.join(out_dir, "X_dev.npz"), X_dev)
    sparse.save_npz(os.path.join(out_dir, "X_test.npz"), X_test)
    np.save(os.path.join(out_dir, "y_train.npy"), y_train)
    np.save(os.path.join(out_dir, "y_dev.npy"), y_dev)
    np.save(os.path.join(out_dir, "y_test.npy"), y_test)
    joblib.dump(vectorizer, os.path.join(out_dir, "vectorizer.joblib"))
    joblib.dump(label_encoder, os.path.join(out_dir, "label_encoder.joblib"))

    print(f"  sizes: {X_train.shape[0]}/{X_dev.shape[0]}/{X_test.shape[0]}")
    print(f"  vocab size: {len(vectorizer.vocabulary_)}")


if __name__ == "__main__":
    os.makedirs(OUT_DIR, exist_ok=True)
    for lang in LANGS:
        for method in ["tfidf", "bow"]:
            process_language(lang, method)

=== eng (tfidf) ===
  sizes: 3309/472/948
  vocab size: 176037
=== eng (bow) ===
  sizes: 3309/472/948
  vocab size: 176037
=== xho (tfidf) ===
  sizes: 1032/147/297
  vocab size: 34012
=== xho (bow) ===
  sizes: 1032/147/297
  vocab size: 34012
=== sna (tfidf) ===
  sizes: 1288/185/369
  vocab size: 44893
=== sna (bow) ===
  sizes: 1288/185/369
  vocab size: 44893


In [10]:
# Task 2 - Multinomial logistic regression implementation - Model

"""
A single linear layer + softmax, implemented as PyTorch nn.Module.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultinomialLogisticRegression(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.linear = nn.Linear(n_features, n_classes)

    def forward(self, X):
        # raw scores, not probabilities yet
        return self.linear(X)

    def compute_probabilities(self, X):
        logits = self.forward(X)
        return F.softmax(logits, dim=1)

    def predict(self, X):
        probs = self.compute_probabilities(X)
        return torch.argmax(probs, dim=1)

In [15]:
# The model class
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultinomialLogisticRegression(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.linear = nn.Linear(n_features, n_classes)

    def forward(self, X):
        return self.linear(X)

    def compute_probabilities(self, X):
        logits = self.forward(X)
        return F.softmax(logits, dim=1)

    def predict(self, X):
        probs = self.compute_probabilities(X)
        return torch.argmax(probs, dim=1)

In [18]:
# Task 3 - Training
import numpy as np
import torch
import torch.nn as nn


def evaluate(model, X, y):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.from_numpy(X.toarray()).float()
        y_tensor = torch.from_numpy(y).long()
        preds = model.predict(X_tensor)
        acc = (preds == y_tensor).float().mean().item()
    return acc


def train(model, X_train, y_train, X_dev, y_dev,
          batch_size=64, lr=0.1, epochs=50, patience=5):

    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    n = X_train.shape[0]
    best_acc = 0
    best_weights = None
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        idx = np.random.permutation(n)  # shuffle the training data each epoch

        for start in range(0, n, batch_size):
            batch = idx[start:start + batch_size]
            X_batch = torch.from_numpy(X_train[batch].toarray()).float()
            y_batch = torch.from_numpy(y_train[batch]).long()

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

        dev_acc = evaluate(model, X_dev, y_dev)
        print(f"epoch {epoch + 1}: dev_acc={dev_acc:.4f}")

        # stop once dev accuracy hasn't improved for a while, instead of
        # always running the full number of epochs - the three languages
        # have very different amounts of training data so they won't
        # converge at the same speed
        if dev_acc > best_acc:
            best_acc = dev_acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("stopping early - dev accuracy stopped improving")
                break

    model.load_state_dict(best_weights)  # keep the best epoch, not the last one
    return model


if __name__ == "__main__":
    import joblib
    from scipy import sparse

    lang, method = "sna", "tfidf"
    X_train = sparse.load_npz(f"features/{lang}/{method}/X_train.npz")
    y_train = np.load(f"features/{lang}/{method}/y_train.npy")
    X_dev = sparse.load_npz(f"features/{lang}/{method}/X_dev.npz")
    y_dev = np.load(f"features/{lang}/{method}/y_dev.npy")
    label_encoder = joblib.load(f"features/{lang}/{method}/label_encoder.joblib")

    model = MultinomialLogisticRegression(
        n_features=X_train.shape[1], n_classes=len(label_encoder.classes_)
    )
    model = train(model, X_train, y_train, X_dev, y_dev)

epoch 1: dev_acc=0.2703
epoch 2: dev_acc=0.3297
epoch 3: dev_acc=0.4811
epoch 4: dev_acc=0.4865
epoch 5: dev_acc=0.4649
epoch 6: dev_acc=0.3676
epoch 7: dev_acc=0.3568
epoch 8: dev_acc=0.5784
epoch 9: dev_acc=0.4811
epoch 10: dev_acc=0.7405
epoch 11: dev_acc=0.7459
epoch 12: dev_acc=0.7568
epoch 13: dev_acc=0.7784
epoch 14: dev_acc=0.7730
epoch 15: dev_acc=0.7838
epoch 16: dev_acc=0.7784
epoch 17: dev_acc=0.7946
epoch 18: dev_acc=0.6541
epoch 19: dev_acc=0.7568
epoch 20: dev_acc=0.7622
epoch 21: dev_acc=0.7622
epoch 22: dev_acc=0.7892
stopping early - dev accuracy stopped improving


In [20]:
#Task 4 - Comparing features

import joblib
import numpy as np
from scipy import sparse


LANGS = ["eng", "xho", "sna"]
METHODS = ["bow", "tfidf"]

# tfidf vectors are L2-normalised (unit length per row), so each feature
# value is much smaller than a raw bow count - the same lr that works for
# bow barely moves tfidf weights per step. Using a higher lr for tfidf
# keeps the comparison fair instead of penalising it for an unrelated
# optimisation-scale issue.
LR = {"bow": 0.1, "tfidf": 1.0}

results = {}

for lang in LANGS:
    for method in METHODS:
        print(f"\n----- {lang} / {method} -----")
        X_train = sparse.load_npz(f"features/{lang}/{method}/X_train.npz")
        y_train = np.load(f"features/{lang}/{method}/y_train.npy")
        X_dev = sparse.load_npz(f"features/{lang}/{method}/X_dev.npz")
        y_dev = np.load(f"features/{lang}/{method}/y_dev.npy")
        label_encoder = joblib.load(f"features/{lang}/{method}/label_encoder.joblib")

        model = MultinomialLogisticRegression(
            n_features=X_train.shape[1], n_classes=len(label_encoder.classes_)
        )
        model = train(model, X_train, y_train, X_dev, y_dev, lr=LR[method])
        dev_acc = evaluate(model, X_dev, y_dev)
        results[(lang, method)] = dev_acc

print("\n=== summary (best dev accuracy) ===")
print(f"{'lang':<6}{'bow':>10}{'tfidf':>10}")
for lang in LANGS:
    print(f"{lang:<6}{results[(lang, 'bow')]:>10.4f}{results[(lang, 'tfidf')]:>10.4f}")


----- eng / bow -----
epoch 1: dev_acc=0.8093
epoch 2: dev_acc=0.7373
epoch 3: dev_acc=0.8199
epoch 4: dev_acc=0.8686
epoch 5: dev_acc=0.8644
epoch 6: dev_acc=0.8750
epoch 7: dev_acc=0.8750
epoch 8: dev_acc=0.8729
epoch 9: dev_acc=0.8750
epoch 10: dev_acc=0.8708
epoch 11: dev_acc=0.8708
stopping early - dev accuracy stopped improving

----- eng / tfidf -----
epoch 1: dev_acc=0.6059
epoch 2: dev_acc=0.5445
epoch 3: dev_acc=0.5890
epoch 4: dev_acc=0.5699
epoch 5: dev_acc=0.7352
epoch 6: dev_acc=0.8093
epoch 7: dev_acc=0.7394
epoch 8: dev_acc=0.7860
epoch 9: dev_acc=0.8242
epoch 10: dev_acc=0.8326
epoch 11: dev_acc=0.8475
epoch 12: dev_acc=0.8496
epoch 13: dev_acc=0.8347
epoch 14: dev_acc=0.8517
epoch 15: dev_acc=0.8496
epoch 16: dev_acc=0.8517
epoch 17: dev_acc=0.8538
epoch 18: dev_acc=0.8496
epoch 19: dev_acc=0.8686
epoch 20: dev_acc=0.8644
epoch 21: dev_acc=0.8559
epoch 22: dev_acc=0.8602
epoch 23: dev_acc=0.8602
epoch 24: dev_acc=0.8623
stopping early - dev accuracy stopped improving